In [ ]:

import os
import sys
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use("Agg")
plt.rcParams["svg.fonttype"] = "none"

sys.path.append("/workspace/experiments/04302026_analysis_data_v2")
from utils import load_adata, load_fitness_data, run_module_analysis, save_module_outputs

ADATA_PATH = "/workspace/experiments/04302026_analysis_data_v2/adata_de122_lce75_merged.h5ad"  # None falls back to the default path defined in utils.py
ADATA_CASE_PATH = (
    "/workspace/experiments/04302026_analysis_data_v2/adata_de122_lce75_merged.case.h5ad"
)
REPRESENTATION_OBSM_KEY = "scvi_latent"
PERTURBATION_OBS_KEY = "target"
CONTROL_PERTURBATION_KEY = "nontargeting"
GLOBAL_SIGMA = 5.0
MODE = "mmd_stat"
THRESHOLD = 0.15
POINT_SIZE = 2.0
UMAP_DPI = 500
N_GENES = 5
LFC_FIGSIZE = (10, 4)
LFC_RANGE = (-3, 3)

adata = load_adata(adata_path=ADATA_PATH)


In [ ]:
import jax.numpy as jnp                                                                                                                                                                                                              
import numpy as np                                                                                                                                                                                                                   
from essential.stats import MMDTestJax                                                                                                                                                                                               
                                                                                                                                                                                                                                    
groups = adata.obs[PERTURBATION_OBS_KEY].unique().tolist()
G = len(groups)                                                                                                                                                                                                                      
                                                                                                                                                                                                                                    
latent = adata.obsm[REPRESENTATION_OBSM_KEY]
counts_list = []
for g in groups:
    mask = adata.obs[PERTURBATION_OBS_KEY] == g
    counts_list.append(mask.sum())

max_n = max(counts_list)
n_features = latent.shape[1]

X_all = np.zeros((G, max_n, n_features), dtype=np.float32)
counts = np.array(counts_list, dtype=np.int32)

for i, g in enumerate(groups):
    mask = adata.obs[PERTURBATION_OBS_KEY] == g
    samples = latent[mask]
    X_all[i, : len(samples)] = samples


mmd = MMDTestJax(kernel_type="rbf", sigma=GLOBAL_SIGMA, max_n=max_n)

dist_matrix = np.array(
    mmd.compute_distance_matrix(jnp.array(X_all), jnp.array(counts), batch_size=1024)
)

In [ ]:
dist_matrix = np.clip(dist_matrix, a_min=0.0, a_max=np.inf)

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
tsne_ = TSNE(metric="precomputed", init="random").fit_transform(dist_matrix)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%matplotlib inline

In [ ]:
plt.scatter(tsne_[:, 0], tsne_[:, 1])
plt.show()